# Source-aware calibration and open-set prototype

This research-only challenger tests two high-priority improvements without changing production artifacts:

1. nested, encounter-grouped calibration under natural source/class prevalence, with source, era, and evidence regime used only in calibration;
2. a hierarchical open-set prototype that estimates modeled ecotype versus Other before estimating SRKW versus Transient.

The notebook writes only to `outputs/improved/`. It cannot certify soft counts because there is no blinded unknown-label audit sample and the known-Other sample is small.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from experiment_support import resolve_release_paths, run_improved_experiment

paths = resolve_release_paths()
output_dir = Path(os.environ['MARINE_MAMMALS_RESEARCH_OUTPUT_ROOT']).expanduser().resolve() / 'improved'
paths

In [ ]:
experiment = run_improved_experiment(output_dir, paths)
experiment["manifest_path"]

## Binary probability quality

All rows below use the same encounter-held-out evaluation units and SOURCE × observed-class post-stratification weights. Tuning is nested inside five outer encounter-grouped folds. The prevalence and transparent local-support rows prevent us from claiming improvement merely by beating a complex incumbent.

In [ ]:
binary = experiment["binary_comparison"].set_index("model")
display(binary.round(5))
display(pd.Series(experiment["bootstrap"], name="value").to_frame().round(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
binary[["brier", "log_loss"]].plot.bar(ax=axes[0], rot=25, title="Natural-prevalence probability loss")
binary[["equal_mass_ece_10"]].plot.bar(ax=axes[1], rot=25, legend=False, title="Equal-mass calibration error")
for axis in axes:
    axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## Source robustness and transport

The first table evaluates within-source performance under cross-fitting. The second withholds each entire source while fitting a pooled calibrator on the remaining sources. A model that improves the first table but degrades the second is learning source-specific prevalence and should not be trusted on a new feed.

In [ ]:
source_brier = experiment["binary_metrics_by_source"].pivot(index="source", columns="model", values="brier")
display(source_brier.round(5))
display(experiment["leave_one_source_out"].round(5))
display(experiment["calibrator_selections"])

## Open-set result

Stage A uses only location, season, coordinate uncertainty, report count, quality tier, and time precision—never source. Stage B uses the nested conditional SRKW probability above. Other recall is evaluated out of fold and its operating threshold is also cross-fitted.

In [ ]:
display(pd.Series(experiment["open_set_metrics"], name="value").to_frame())
display(experiment["multiclass_comparison"].set_index("model").round(5))
display(experiment["open_set_selections"])

## Decision

This experiment is deliberately fail-closed. A binary calibration improvement is not enough to release soft counts. Promotion still requires a blinded, double-reviewed unknown-label audit, adequate Other support, rolling-origin and repeated spatial evaluation, source/era gates, stability checks on every counted row, and immutable model/scorer/evidence binding.